# Chapter 25 — Numerical Methods for Economists

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch25_numerical_methods.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 25.1 Numbers inside a computer

In [ ]:
import numpy as np

print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

In [ ]:
eps = np.finfo(float).eps
print(f"machine epsilon: {eps:.3e}")
print("1 + eps/2 == 1 ?", 1 + eps / 2 == 1)

# A rupee example: add 10 paise one lakh times
total = 0.0
for _ in range(100_000):
    total += 0.10
print(f"one lakh additions of Rs 0.10 = {total!r}")
print(f"error = {total - 10_000:.3e} rupees")

In [ ]:
print(np.isclose(0.1 + 0.2, 0.3))
print(np.isclose(total, 10_000))

## 25.2 The data: India's onion price cycles

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from indiadata import mospi, mospi_cpi_groups

parts = []
for year in range(2013, 2026):
    raw = mospi("cpi/getItemIndex", year=year)            # cached in data/raw
    print(year, sorted(raw["baseyear"].unique()), end=" | ")
    raw = raw[(raw["baseyear"] == "2012") & raw["item"].isin(["Onion", "Potato"])]
    parts.append(raw)
items = pd.concat(parts, ignore_index=True)
items["date"] = pd.to_datetime(items["year"] + " " + items["month"],
                               format="%Y %B")
items["index"] = pd.to_numeric(items["index"])
assert not items.duplicated(["item", "date"]).any()
veg_items = items.pivot(index="date", columns="item", values="index")
del parts, raw, items

groups = mospi_cpi_groups("combined")
veg = groups.loc[groups["subgroup"] == "Vegetables"].set_index("date")["index"]
del groups
prices = veg_items.join(veg.rename("Vegetables"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(prices.index, prices["Vegetables"], color="#9db7d5", linewidth=2,
        label="Vegetables (sub-group)")
ax.plot(prices.index, prices["Potato"], color="#c55a11", linewidth=1.5,
        label="Potato")
ax.plot(prices.index, prices["Onion"], color="#1f4e79", linewidth=2,
        label="Onion")
ax.set_ylabel("CPI-Combined, 2012 = 100")
ax.set_title("Onion prices spike far more than other vegetables", loc="left",
             fontsize=13)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.04, 1, 1))
fig.text(0.01, 0.005, "Source: MoSPI, CPI (base 2012 = 100), item and "
         "sub-group indices, All India combined, via eSankhyiki API.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
onion = prices["Onion"]
peak = onion.idxmax()
same_month_before = peak - pd.DateOffset(years=1)
ratio_2019 = onion[peak] / onion[same_month_before]
print(f"Peak onion index: {onion[peak]:.1f} in {peak:%B %Y}")
print(f"Same month a year earlier: {onion[same_month_before]:.1f}")
print(f"Price ratio: {ratio_2019:.2f}")
print(f"Month-to-month volatility (s.d. of % change): "
      f"onion {onion.pct_change().std():.1%}, "
      f"potato {prices['Potato'].pct_change().std():.1%}, "
      f"vegetables {prices['Vegetables'].pct_change().std():.1%}")

## 25.3 A model with no formula for its solution

In [ ]:
# ILLUSTRATIVE ASSUMPTIONS (not estimated in this chapter)
OMEGA, EPS1, EPS2, ETA = 0.5, 0.5, 0.1, 0.1


def demand(p):
    return OMEGA * p ** -EPS1 + (1 - OMEGA) * p ** -EPS2


def supply(p, s=0.0):
    return (1 - s) * p ** ETA


def excess_demand(p, s=0.0):
    return demand(p) - supply(p, s)


print("Z(1) in a normal year:", excess_demand(1.0))
print("Z(1) after a 20% supply loss:", round(excess_demand(1.0, 0.2), 4))

## 25.4 Bisection, by hand

In [ ]:
def bisection(f, a, b, tol=1e-10, max_iter=200):
    """Root of f in [a, b] by bisection. Returns root and the history."""
    fa = f(a)
    if fa * f(b) > 0:
        raise ValueError("f(a) and f(b) must have opposite signs")
    history = []
    for _ in range(max_iter):
        m = (a + b) / 2
        fm = f(m)
        history.append(m)
        if abs(fm) < tol or (b - a) / 2 < tol:
            break
        if fm * fa > 0:          # same sign as f(a): root is to the right
            a, fa = m, fm
        else:
            b = m
    return m, history


S_SHOCK = 0.2
p_bis, hist_bis = bisection(lambda p: excess_demand(p, S_SHOCK), 1.0, 10.0)
print(f"equilibrium price: {p_bis:.8f}  (normal year = 1)")
print(f"iterations: {len(hist_bis)}")
print(f"excess demand there: {excess_demand(p_bis, S_SHOCK):.1e}")

## 25.5 Newton's method, by hand

In [ ]:
def dZ(p, s=0.0):
    return (-OMEGA * EPS1 * p ** (-EPS1 - 1)
            - (1 - OMEGA) * EPS2 * p ** (-EPS2 - 1)
            - (1 - s) * ETA * p ** (ETA - 1))


def newton(f, fprime, x0, tol=1e-10, max_iter=50):
    """Root of f by Newton's method from starting value x0."""
    x, history = x0, [x0]
    for _ in range(max_iter):
        step = f(x) / fprime(x)
        x = x - step
        history.append(x)
        if abs(step) < tol:
            break
    return x, history


p_newt, hist_newt = newton(lambda p: excess_demand(p, S_SHOCK),
                           lambda p: dZ(p, S_SHOCK), x0=1.0)
print(f"Newton: {p_newt:.8f} after {len(hist_newt) - 1} iterations")
for k, x in enumerate(hist_newt):
    print(f"  step {k}: p = {x:.12f}   error = {abs(x - p_newt):.1e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3))
grid = np.linspace(0.8, 3.0, 200)
ax1.plot(grid, excess_demand(grid, S_SHOCK), color="#1f4e79", linewidth=2)
ax1.axhline(0, color="black", linewidth=0.8)
ax1.plot(p_bis, 0, "o", color="#c55a11")
ax1.annotate(f"p* = {p_bis:.3f}", (p_bis, 0), xytext=(p_bis + 0.25, 0.08),
             color="#c55a11", arrowprops=dict(arrowstyle="-", color="grey"))
ax1.set_xlabel("price (normal year = 1)")
ax1.set_ylabel("excess demand Z(p)")
ax1.set_title("Excess demand after a 20% supply loss", loc="left",
              fontsize=11)

err_bis = np.abs(np.array(hist_bis) - p_newt) + 1e-17
err_newt = np.abs(np.array(hist_newt[:-1]) - p_newt) + 1e-17
ax2.semilogy(range(len(err_bis)), err_bis, "o-", color="#9db7d5",
             markersize=3, label="bisection")
ax2.semilogy(range(len(err_newt)), err_newt, "o-", color="#c55a11",
             label="Newton")
ax2.set_xlabel("iteration")
ax2.set_ylabel("absolute error (log scale)")
ax2.set_title("Convergence: linear vs quadratic", loc="left", fontsize=11)
ax2.legend(frameon=False)
for a in (ax1, ax2):
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Illustrative model: elasticities are assumptions, "
         "not estimates (see section 25.3).", fontsize=8, color="grey")
plt.show()

In [ ]:
with np.errstate(invalid="ignore"):
    _, bad = newton(lambda p: excess_demand(p, S_SHOCK),
                    lambda p: dZ(p, S_SHOCK), x0=np.float64(5.0), max_iter=4)
print([round(float(x), 3) for x in bad])

## 25.6 SciPy's root finders

In [ ]:
from scipy import optimize

p_brent, info = optimize.brentq(excess_demand, 1.0, 10.0, args=(S_SHOCK,),
                                xtol=1e-12, full_output=True)
print(f"brentq:  {p_brent:.10f}  ({info.iterations} iterations, "
      f"{info.function_calls} function calls)")

res = optimize.root_scalar(excess_demand, args=(S_SHOCK,), x0=1.0,
                           fprime=dZ, method="newton")
print(f"root_scalar (newton): {res.root:.10f}  ({res.iterations} iterations)")
print("all three agree:", np.isclose(p_brent, p_bis) and np.isclose(p_brent, res.root))

### Comparative statics: from shock to price, and back

In [ ]:
shocks = np.linspace(0, 0.5, 11)
p_star = [optimize.brentq(excess_demand, 0.5, 50, args=(s,)) for s in shocks]
cs_table = pd.DataFrame({"supply loss": shocks, "price (normal = 1)": p_star})
cs_table["price rise, %"] = 100 * (cs_table["price (normal = 1)"] - 1)
cs_table.round(3).iloc[::2]

In [ ]:
def equilibrium_price(s):
    return optimize.brentq(excess_demand, 0.5, 1e4, args=(s,))


implied_s = optimize.brentq(lambda s: equilibrium_price(s) - ratio_2019,
                            0.0, 0.7)
print(f"supply loss that reproduces a {ratio_2019:.2f}x price rise: "
      f"{implied_s:.1%}")

# One-group benchmark, which has a formula: (1 - s) = p^-(eps + eta)
for e_one in (EPS1, EPS2):
    s_formula = 1 - ratio_2019 ** -(e_one + ETA)
    print(f"  with a single elasticity of {e_one}: {s_formula:.1%}")

## 25.7 Systems of equations: onions and potatoes

In [ ]:
# ILLUSTRATIVE ASSUMPTIONS: own- and cross-price elasticities
OWN = {"onion": 0.3, "potato": 0.4}
CROSS = {"onion": 0.05, "potato": 0.15}   # response to the *other* price


def excess_two(p, s_onion=0.0, s_potato=0.0):
    po, pp = p
    z_o = po ** -OWN["onion"] * pp ** CROSS["onion"] - (1 - s_onion) * po ** ETA
    z_p = pp ** -OWN["potato"] * po ** CROSS["potato"] - (1 - s_potato) * pp ** ETA
    return [z_o, z_p]


sol, info, ok, msg = optimize.fsolve(excess_two, x0=[1.0, 1.0], args=(0.2,),
                                     full_output=True)
print("converged:", ok == 1, "|", msg)
print(f"onion price {sol[0]:.3f}, potato price {sol[1]:.3f}")
print("residuals:", np.round(excess_two(sol, 0.2), 12))

## 25.8 Numerical derivatives

In [ ]:
x0 = 1.5
exact = -OMEGA * EPS1 * x0 ** (-EPS1 - 1) - (1 - OMEGA) * EPS2 * x0 ** (-EPS2 - 1)
steps = 10.0 ** -np.arange(1, 16)
fwd = np.abs((demand(x0 + steps) - demand(x0)) / steps - exact)
cen = np.abs((demand(x0 + steps) - demand(x0 - steps)) / (2 * steps) - exact)
best_f, best_c = steps[fwd.argmin()], steps[cen.argmin()]
print(f"exact D'(1.5) = {exact:.10f}")
print(f"best step: forward h = {best_f:.0e} (error {fwd.min():.1e}), "
      f"central h = {best_c:.0e} (error {cen.min():.1e})")
print(f"rule of thumb: sqrt(eps) = {np.sqrt(eps):.1e}, cbrt(eps) = {eps ** (1/3):.1e}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(steps, fwd, "o-", color="#1f4e79", label="forward difference")
ax.loglog(steps, cen, "s-", color="#c55a11", label="central difference")
ax.invert_xaxis()
ax.set_xlabel("step size h (log scale, smaller to the right)")
ax.set_ylabel("absolute error (log scale)")
ax.set_title("The step-size trade-off in numerical derivatives", loc="left",
             fontsize=12)
ax.text(1e-2, 1e-12, "truncation error\nfalls with h", fontsize=9,
        color="grey", ha="center")
ax.text(1e-12, 1e-10, "rounding error\ngrows as h shrinks", fontsize=9,
        color="grey", ha="center")
ax.legend(frameon=False, loc="upper center")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Derivative of the illustrative demand function at "
         "p = 1.5; exact value known analytically.", fontsize=8, color="grey")
plt.show()

In [ ]:
def central_diff(f, x, h=None):
    h = eps ** (1 / 3) * max(abs(x), 1.0) if h is None else h
    return (f(x + h) - f(x - h)) / (2 * h)


elast = central_diff(demand, p_brent) * p_brent / demand(p_brent)
dp_ds_numeric = central_diff(equilibrium_price, S_SHOCK)
# implicit function theorem: dp*/ds = -(dZ/ds)/(dZ/dp), with dZ/ds = p^eta
dp_ds_ift = -(p_brent ** ETA) / dZ(p_brent, S_SHOCK)
print(f"market demand elasticity at p* = {elast:.3f}")
print(f"dp*/ds by finite differences:    {dp_ds_numeric:.5f}")
print(f"dp*/ds by implicit function thm: {dp_ds_ift:.5f}")

## 25.9 Numerical integration: the cost of a price spike

In [ ]:
def trapezoid(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (y[0] / 2 + y[1:-1].sum() + y[-1] / 2)


def simpson(f, a, b, n):
    if n % 2:
        raise ValueError("Simpson's rule needs an even number of intervals")
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h / 3 * (y[0] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum() + y[-1])


def demand_integral_exact(a, b):
    F = lambda p: (OMEGA * p ** (1 - EPS1) / (1 - EPS1)
                   + (1 - OMEGA) * p ** (1 - EPS2) / (1 - EPS2))
    return F(b) - F(a)


a, b = 1.0, ratio_2019
exact_cs = demand_integral_exact(a, b)
rows = []
for n in (2, 4, 8, 16, 32, 64):
    rows.append({"n": n,
                 "trapezoid error": abs(trapezoid(demand, a, b, n) - exact_cs),
                 "Simpson error": abs(simpson(demand, a, b, n) - exact_cs)})
from scipy import integrate
quad_val, quad_err = integrate.quad(demand, a, b)
print(f"exact: {exact_cs:.10f} | quad: {quad_val:.10f} "
      f"(error estimate {quad_err:.1e})")
pd.DataFrame(rows).set_index("n").map("{:.2e}".format)

In [ ]:
print(f"consumer-surplus loss: {quad_val:.2f} x normal annual onion spending")
q_new = demand(ratio_2019)
print(f"quantity bought at the peak price: {q_new:.3f} of normal")
print(f"spending at the peak price: {ratio_2019 * q_new:.2f} x normal")
print(f"extra cost of the old quantity: {ratio_2019 - 1:.2f} x normal")

## 25.10 Who pays a tax? Incidence, numerically

In [ ]:
T_RATE = 0.18
ETA_LR = 1.0                                  # illustrative assumption


def tax_gap(pc, t, eta):
    return demand(pc) - (pc / (1 + t)) ** eta


pc = optimize.brentq(tax_gap, 1.0, 1 + T_RATE, args=(T_RATE, ETA_LR))
pp = pc / (1 + T_RATE)
wedge = pc - pp
consumer_share = (pc - 1) / wedge
avg_eps = OMEGA * EPS1 + (1 - OMEGA) * EPS2
print(f"consumer price {pc:.4f}, producer price {pp:.4f}")
print(f"consumers bear {consumer_share:.1%} of the tax")
print(f"formula with the average elasticity {avg_eps}: "
      f"{ETA_LR / (ETA_LR + avg_eps):.1%}")

# deadweight loss: area between demand and supply from Q_tax to Q_0 = 1
q_tax = demand(pc)
inv_demand = lambda q: optimize.brentq(lambda p: demand(p) - q, 1e-3, 1e3)
inv_supply = lambda q: q ** (1 / ETA_LR)
dwl, _ = integrate.quad(lambda q: inv_demand(q) - inv_supply(q), q_tax, 1.0)
revenue = (pc - pp) * q_tax
print(f"tax revenue {revenue:.4f}, deadweight loss {dwl:.5f} "
      f"({dwl / revenue:.1%} of revenue)")

In [ ]:
qs = np.linspace(0.6, 1.25, 200)
ps_d = [inv_demand(q) for q in qs]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(qs, ps_d, color="#1f4e79", linewidth=2, label="demand (two groups)")
ax.plot(qs, inv_supply(qs), color="#c55a11", linewidth=2,
        label="supply, before tax")
ax.plot(qs, (1 + T_RATE) * inv_supply(qs), color="#c55a11", linestyle="--",
        linewidth=1.5, label="supply incl. 18% tax")
qq = np.linspace(q_tax, 1.0, 50)
ax.fill_between(qq, [inv_demand(q) for q in qq], inv_supply(qq),
                color="#9db7d5", alpha=0.8, label="deadweight loss")
ax.hlines([pc, pp], 0.6, q_tax, colors=["#1f4e79", "#c55a11"],
          linestyles=":", linewidth=1.2)
ax.text(0.61, pc + 0.015, f"consumers pay {pc:.3f}", color="#1f4e79",
        fontsize=9)
ax.text(0.61, pp - 0.04, f"producers receive {pp:.3f}", color="#c55a11",
        fontsize=9)
ax.set_xlim(0.6, 1.25)
ax.set_ylim(0.6, 1.6)
ax.set_xlabel("quantity (no-tax equilibrium = 1)")
ax.set_ylabel("price (no-tax equilibrium = 1)")
ax.set_title("Incidence of an ad valorem tax", loc="left", fontsize=12)
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Illustrative elasticities (section 25.3) and "
         "eta = 1. Prices and quantities relative to the no-tax equilibrium.",
         fontsize=8, color="grey")
plt.show()

## 25.11 Answering the question

## Key ideas

## Exercises

## Further reading